# Applied AI — End-to-End Sony SampleID + DTW Refinement

This notebook runs the whole experimental pipeline in one place:

**Upload source + derivative → Sony embeddings → rank + temporal sequence grouping → DTW refinement of every placement → coverage / pitch / timestamp summary → CSV downloads**

> This is an experimental measurement pipeline, not a legal conclusion. Sony finds candidate regions; DTW refines timing. Validate against known human annotations before relying on the results.

In [ ]:
# 1) Install the project + Sony SampleID
!apt-get -qq update
!apt-get -qq install -y ffmpeg
!rm -rf /content/projects /content/sampleid
!git clone -q https://github.com/manzaa-glitch/projects.git /content/projects
!git clone -q https://github.com/sony/sampleid.git /content/sampleid
!pip -q install -e /content/sampleid librosa pandas scipy soundfile tqdm

import sys
sys.path.insert(0, '/content/projects')
sys.path.insert(0, '/content/sampleid')
sys.path.insert(0, '/content/sampleid/sampleid')

print('Setup complete.')

In [ ]:
# 2) Mount Google Drive and locate Sony's pretrained checkpoint
from google.colab import drive
from pathlib import Path
import shutil, zipfile, requests

drive.mount('/content/drive')

DRIVE_CKPT = Path('/content/drive/MyDrive/AppliedAI/sampleid-best.ckpt')
LOCAL_CKPT = Path('/content/sampleid-best.ckpt')

def valid_checkpoint(path):
    return path.exists() and path.stat().st_size > 10_000_000 and zipfile.is_zipfile(path)

if valid_checkpoint(DRIVE_CKPT):
    print('Found checkpoint in Google Drive. Copying to Colab runtime...')
    shutil.copy2(DRIVE_CKPT, LOCAL_CKPT)
else:
    print('Checkpoint not found at MyDrive/AppliedAI/sampleid-best.ckpt.')
    print('Trying Zenodo as a fallback. If Zenodo is unavailable, place sampleid-best.ckpt in that Drive folder and rerun this cell.')
    urls = [
        'https://zenodo.org/api/records/17413869/files/sampleid-best.ckpt/content',
        'https://zenodo.org/records/17413869/files/sampleid-best.ckpt?download=1',
    ]
    last_error = None
    for url in urls:
        try:
            with requests.get(url, stream=True, timeout=180, headers={'User-Agent':'Mozilla/5.0'}) as r:
                r.raise_for_status()
                with open(LOCAL_CKPT, 'wb') as f:
                    for chunk in r.iter_content(chunk_size=1024*1024):
                        if chunk:
                            f.write(chunk)
            if valid_checkpoint(LOCAL_CKPT):
                print('Zenodo download succeeded.')
                break
        except Exception as e:
            last_error = e
            LOCAL_CKPT.unlink(missing_ok=True)

if not valid_checkpoint(LOCAL_CKPT):
    raise RuntimeError(
        'Sony checkpoint unavailable. Save sampleid-best.ckpt to Google Drive at '
        'MyDrive/AppliedAI/sampleid-best.ckpt and rerun cell 2. '
        f'Latest Zenodo error: {last_error}'
    )

print(f'Checkpoint ready: {LOCAL_CKPT.stat().st_size/1e6:.1f} MB')

In [ ]:
# 3) Load Sony model + pipeline functions
import torch
from sampleid import SampleID
from sony_sampleid.core import analyze_with_model
from dtw_refinement.batch_refine import refine_sony_detail

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SampleID.load_checkpoint(ckpt_path=str(LOCAL_CKPT), device=device)
model.eval()
print('Sony model loaded on', device)

In [ ]:
# 4) Upload ORIGINAL / SOURCE song
from google.colab import files
print('Choose the original/source song:')
up = files.upload()
source_path = next(iter(up.keys()))
print('Source:', source_path)

In [ ]:
# 5) Upload SAMPLED / DERIVATIVE song
print('Choose the sampled/derivative song:')
up = files.upload()
derivative_path = next(iter(up.keys()))
print('Derivative:', derivative_path)

In [ ]:
# 6) Run Sony candidate localization + grouping
# You can tune these later. These are the same defaults used in the app.
CHUNK_SECONDS = 5.0
HOP_SECONDS = 2.5
TOP_SOURCE_CANDIDATES = 3
MIN_SEQUENCE_SUPPORT = 3

candidates_df, sony_summary_df, sony_detail_df, diagnostics = analyze_with_model(
    model,
    source_path,
    derivative_path,
    chunk_seconds=CHUNK_SECONDS,
    hop_seconds=HOP_SECONDS,
    top_k=TOP_SOURCE_CANDIDATES,
    min_support=MIN_SEQUENCE_SUPPORT,
)

print('Raw pairwise comparisons:', f"{diagnostics['num_pairwise_scores']:,}")
print('Raw cosine range:', round(diagnostics['similarity_min'],4), 'to', round(diagnostics['similarity_max'],4))
print('Grouped Sony candidate regions:', len(sony_summary_df))

display(sony_summary_df)

if sony_detail_df.empty:
    raise RuntimeError('Sony grouping found no coherent candidate placements with the current settings.')

In [ ]:
# 7) Automatically run DTW on EVERY Sony placement
DTW_SEARCH_PADDING = 8.0
DTW_MAX_PITCH_SHIFT = 6

def progress(done, total):
    if total:
        print(f'\rDTW placements: {done}/{total}', end='')

refined_detail_df, refined_group_df = refine_sony_detail(
    source_path,
    derivative_path,
    sony_detail_df,
    search_padding_s=DTW_SEARCH_PADDING,
    max_pitch_shift_semitones=DTW_MAX_PITCH_SHIFT,
    progress_callback=progress,
)
print('\nDTW complete.')
display(refined_group_df)


In [ ]:
# 8) Build an overall measurement summary
import librosa, pandas as pd

def union_duration(intervals):
    spans = sorted((float(s), float(e)) for s,e in intervals if e > s)
    if not spans:
        return 0.0
    total = 0.0
    s0, e0 = spans[0]
    for s,e in spans[1:]:
        if s <= e0:
            e0 = max(e0, e)
        else:
            total += e0 - s0
            s0, e0 = s,e
    return total + (e0-s0)

ok = refined_detail_df[refined_detail_df['status'] == 'ok'].copy()
derivative_duration = float(librosa.get_duration(path=derivative_path))

intervals = list(zip(
    ok['refined_derivative_start_s'].astype(float),
    ok['refined_derivative_end_s'].astype(float),
)) if len(ok) else []

overall_detected_seconds = union_duration(intervals)
overall_coverage_pct = 100.0 * overall_detected_seconds / max(derivative_duration, 1e-9)

summary = pd.DataFrame([{
    'derivative_duration_s': round(derivative_duration, 3),
    'sony_source_groups': int(len(sony_summary_df)),
    'sony_placements_sent_to_dtw': int(len(sony_detail_df)),
    'dtw_placements_refined': int(len(ok)),
    'dtw_failures': int(len(refined_detail_df) - len(ok)),
    'overall_unique_refined_seconds': round(overall_detected_seconds, 3),
    'overall_coverage_percent': round(overall_coverage_pct, 2),
}])

display(summary)
print('\nIMPORTANT: coverage is the union of DTW-refined candidate placements, not a verified legal Nature of Use figure yet.')

In [ ]:
# 9) Review all refined placements
cols = [
    'group','occurrence','source_start','source_end',
    'derivative_start','derivative_end',
    'refined_derivative_start','refined_derivative_end',
    'refined_duration_s','pitch_shift_semitones','dtw_cost','status'
]
cols = [c for c in cols if c in refined_detail_df.columns]
display(refined_detail_df[cols])

In [ ]:
# 10) Download all outputs
outputs = {
    '01_sony_grouped_summary.csv': sony_summary_df,
    '02_sony_detailed_placements.csv': sony_detail_df,
    '03_dtw_refined_group_summary.csv': refined_group_df,
    '04_dtw_refined_placements.csv': refined_detail_df,
    '05_overall_measurement_summary.csv': summary,
}

for filename, df in outputs.items():
    df.to_csv(filename, index=False)

import zipfile
zip_name = 'applied_ai_sony_dtw_results.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as z:
    for filename in outputs:
        z.write(filename)

files.download(zip_name)

## What each stage means

- **Sony grouped summary:** likely source excerpts and repeated placements based on embedding rank + time consistency.
- **Sony detailed placements:** each individual candidate placement passed into DTW.
- **DTW refined placements:** tighter derivative timestamps, rough pitch shift, duration ratio, alignment cost.
- **Overall coverage:** union of all successful refined derivative intervals, so overlapping candidates are not double-counted.

### Current limitation
DTW still treats Sony's source region as the query excerpt. It refines the derivative side but does not yet shrink a rough source region to the exact sampled sub-fragment.